# 03 — Vector Databases with ChromaDB

This notebook builds a persistent knowledge base using ChromaDB.

**What you'll learn:**
- Storing embeddings in ChromaDB (ephemeral and persistent)
- Querying by similarity
- Filtering results with metadata
- The complete indexing + retrieval pipeline

## 1. Setup and imports

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, 'src')

from rag_pipeline.chunking import load_directory, chunk_documents
from rag_pipeline.embeddings import load_model, embed_texts
from rag_pipeline.vectorstore import (
    get_client, get_or_create_collection,
    add_documents, query_collection, collection_stats,
)

print('All imports successful')

## 2. Build the index: Load → Chunk → Embed → Store

In [ ]:
# Step 1: Load documents
docs = load_directory('data/sample/')
print(f'Loaded {len(docs)} documents')

# Step 2: Chunk
chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=50)
print(f'Split into {len(chunks)} chunks')

# Step 3: Embed
model = load_model()
embeddings = embed_texts(model, [c.page_content for c in chunks])
print(f'Embedded → shape {embeddings.shape}')

In [ ]:
# Step 4: Store in ChromaDB (persistent)
client = get_client(persist_dir='./chroma_db')

# Delete existing collection if re-running to avoid duplicates
try:
    client.delete_collection('company_docs')
    print('Deleted existing collection')
except ValueError:
    pass

collection = get_or_create_collection(client, name='company_docs')

# Metadata values must be str, int, float, or bool for ChromaDB
for chunk in chunks:
    for key, val in list(chunk.metadata.items()):
        if not isinstance(val, (str, int, float, bool)):
            chunk.metadata[key] = str(val)

ids = add_documents(collection, chunks, embeddings.tolist())
print(f'\nStored {len(ids)} entries')
print(f'Collection stats: {collection_stats(collection)}')

## 3. Query the knowledge base

In [ ]:
queries = [
    'What is the remote work policy?',
    'How do I deploy to production?',
    'How much PTO do I get?',
    'What is the code review process?',
]

for query in queries:
    query_emb = model.encode(query).tolist()
    results = query_collection(collection, query_emb, top_k=2)

    print(f'\n{"=" * 60}')
    print(f'Query: "{query}"\n')
    for i, r in enumerate(results, 1):
        sim = 1 - r['distance']
        source = r['metadata']['source'].split('/')[-1]
        chunk_idx = r['metadata'].get('chunk_index', '?')
        print(f'  #{i} (similarity: {sim:.3f}) [{source}, chunk {chunk_idx}]')
        print(f'  {r["document"][:120]}...\n')

## 4. Metadata filtering

Search only specific sources — the key production pattern.

In [ ]:
query = 'What are the security requirements?'
query_emb = model.encode(query).tolist()

# Search ALL sources
all_results = query_collection(collection, query_emb, top_k=3)
print('All sources:')
for r in all_results:
    source = r['metadata']['source'].split('/')[-1]
    print(f'  [{1-r["distance"]:.3f}] {source}: {r["document"][:80]}...')

# Search ONLY the engineering wiki
wiki_sources = [c.metadata['source'] for c in chunks if 'engineering' in c.metadata['source']]
if wiki_sources:
    wiki_results = query_collection(
        collection, query_emb, top_k=3,
        where={'source': wiki_sources[0]},
    )
    print(f'\nEngineering wiki only:')
    for r in wiki_results:
        print(f'  [{1-r["distance"]:.3f}] {r["document"][:80]}...')

## 5. Verify persistence

Create a fresh client pointing to the same directory — data should still be there.

In [ ]:
# Simulate a restart: new client, same path
fresh_client = get_client(persist_dir='./chroma_db')
fresh_collection = get_or_create_collection(fresh_client, name='company_docs')

print(f'Reopened collection: {fresh_collection.count()} entries')

# Query without re-embedding documents
test_query = 'What holidays does the company observe?'
test_emb = model.encode(test_query).tolist()
results = query_collection(fresh_collection, test_emb, top_k=2)

print(f'\nQuery: "{test_query}"')
for r in results:
    print(f'  [{1-r["distance"]:.3f}] {r["document"][:100]}...')

## 6. Relevance comparison: related vs. unrelated queries

In [ ]:
related_query = 'What benefits does the company offer?'
unrelated_query = 'How to bake a chocolate cake?'

for label, q in [('Related', related_query), ('Unrelated', unrelated_query)]:
    emb = model.encode(q).tolist()
    results = query_collection(collection, emb, top_k=3)
    avg_sim = sum(1 - r['distance'] for r in results) / len(results)
    top_sim = 1 - results[0]['distance']
    print(f'{label:>10} query: "{q}"')
    print(f'           Top similarity: {top_sim:.3f}, Avg top-3: {avg_sim:.3f}')
    print(f'           Best match: {results[0]["document"][:80]}...\n')

## Key Takeaways

1. **ChromaDB** stores vectors + text + metadata together, persists to disk
2. **Index once, query many times** — no re-embedding needed after the first run
3. **Metadata filtering** narrows search to specific sources/pages before ranking
4. **Distance = 1 − similarity** — lower distance means more relevant
5. Unrelated queries produce noticeably higher distances (lower similarity)

**Next:** We'll connect this retrieval system to an LLM to build the complete
RAG pipeline — asking questions and getting natural-language answers with citations.